In [2]:
# 허깅페이스 로그인
# 주피터 노트북 웹 UI 상에 로그인 위젯 띄우기
from huggingface_hub import notebook_login
notebook_login()

In [3]:
import os
from dotenv import load_dotenv
from typing import List
from typing_extensions import TypedDict
from langgraph.graph import END, StateGraph
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer,pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline

load_dotenv()

embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
vector_db = Chroma(persist_directory="./chroma_db_session", embedding_function=embeddings)
retriever = vector_db.as_retriever(search_kwargs={"k": 2})


hf_model = 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct'
# 4bit 양자화 설정(VRAM절약)
q_config =  BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
# 토크나이져 , 모델 로드
model = AutoModelForCausalLM.from_pretrained(
    hf_model,
    quantization_config = q_config,
    device_map="auto",
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(hf_model,trust_remote_code=True)

# 파이프라인생성 langchain llm 매핑
pipe =  pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True
)
exaone_llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

c:\Users\Playdata\miniconda3\envs\torch_env\lib\site-packages\bitsandbytes\backends\default\ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\Playdata\miniconda3\envs\torch_env\lib\site-packages\bitsandbytes\backends\cpu\ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [4]:
# 로컬모델은 특수기호에 민감해.. 지키지 않으면 환각
from langchain_core.prompts import PromptTemplate
exaone_template = '''[|system|]
당신은 사내 문서를 바탕으로 질문에 답하는 어시스턴트입니다.
아래 [문서] 내용만 참고하고, 없으면 '모릅니다'라고 하세요 [|endofturn|]
[|user|]
[문서]
{context}

질문: {question}[|endofturn|]
[|assistant|]
'''
exaone_prompt = PromptTemplate(
    input_variables=['context','question'],
    template=exaone_template
)
# chain = exaone_prompt | exaone_llm | Strout

In [ ]:
# 1. State 정의: 노드 간에 전달될 데이터 구조
class GraphState(TypedDict):
    question: str
    documents: List[str]
    generation: str
    web_search_needed: bool

# 2. Node 정의
def retrieve(state: GraphState):
    print("  -> [Node] Retrieve: 문서 검색 중...")
    question = state["question"]
    docs = retriever.invoke(question)    
    return {"documents": docs, "question": question}

def grade_documents(state: GraphState):
    print("  -> [Node] Grade: 검색된 문서의 관련성 자체 평가 중...")
    question = state["question"]
    docs = state["documents"]    
    
    prompt = ChatPromptTemplate.from_template(
    "다음 문서가 사용자의 질문과 관련이 있는지 평가하세요.\n"
    "관련이 있으면 'yes', 없으면 'no'라고만 답하세요.\n\n"
    "문서: {context}\n질문: {question}"
    )

    chain = prompt | exaone_llm | StrOutputParser()
    
    filtered_docs = []
    web_search_needed = False
    
    for idx, doc in enumerate(docs):
        score = chain.invoke({"context": doc.page_content, "question": question}).strip().lower()
        if "yes" in score:
            print(f"     - 문서 {idx+1}: [PASS] 관련성 높음")
            filtered_docs.append(doc)
        else:
            print(f"     - 문서 {idx+1}: [FAIL] 관련성 낮음 (환각 위험)")
            web_search_needed = True
            
    if not filtered_docs:
        web_search_needed = True
        
    return {"documents": filtered_docs, "web_search_needed": web_search_needed}

def rewrite_query(state: GraphState):
    print("  -> [Node] Rewrite: 관련 문서 부족으로 쿼리 재작성(폴백) 수행 중...")
    question = state["question"]
    
    prompt = ChatPromptTemplate.from_template(
        "다음 질문을 검색 엔진이나 외부 웹 지식에서 더 잘 찾을 수 있도록 구체적으로 재작성하세요.\n"
        "원래 질문: {question}\n재작성된 질문:"
    )

    chain = prompt | exaone_llm | StrOutputParser()
    better_question = chain.invoke({"question": question})
    print(f"     - 원본 쿼리: {question}")
    print(f"     - 개선된 쿼리: {better_question}")
    return {"question": better_question}

def generate(state: GraphState):
    print("  -> [Node] Generate: 최종 응답 생성 중...")
    question = state["question"]
    docs = state["documents"]
    if docs:
        context = "\n\n".join([doc.page_content for doc in docs])   
    
        prompt = ChatPromptTemplate.from_template(
            "다음 문맥을 바탕으로 질문에 답하세요.\n\n문맥: {context}\n질문: {question}"
        )
        chain = prompt | exaone_llm | StrOutputParser()
        generation = chain.invoke({"context": context, "question": question})
    
        for doc in docs:
            filename = doc.metadata.get('filename','N/A')
            f_type = doc.metadata.get('file_type','N/A')
            cat = doc.metadata.get('category','N/A')
            generation += f'\n\n -출처:{filename}  파일 타입:{f_type}  카테고리:{cat}'
    else:
        generation = '관련 문서를 찾지못했습니다. (외부 지식으로 대처)'

    return {"generation": generation}
# 그래프컴파일 및 엣지 연결
def decide_to_generate(state: GraphState):
    if state['documents'] and state["web_search_needed"]:
        print("  -> [Condition] 일부 문서가 부적합하나 적합한 문서들이 있어 해당 문서를 기반으로 Generate 노드로 분기합니다.")
        return "generate"
    elif state["web_search_needed"]:
        print("  -> [Condition] 일부 문서가 부적합하여 Rewrite 노드로 분기합니다.")
        return "rewrite_query"
    else:
        print("  -> [Condition] 문서가 완벽히 관련성이 있어 Generate 노드로 분기합니다.")
        return "generate"

workflow = StateGraph(GraphState)
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("rewrite_query", rewrite_query)
workflow.add_node("generate", generate)

# 노드 흐름 연결
workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges(
    "grade_documents", 
    decide_to_generate, 
    {"rewrite_query": "rewrite_query", "generate": "generate"}
)
workflow.add_edge("rewrite_query", "generate")
workflow.add_edge("generate", END)

app = workflow.compile()

In [ ]:
print("\n\n[테스트] RAG에대해서 알려주세요 - 문서에 있음")
result = app.invoke({"question": "RAG에 대해서 알려주세요"})
print(f"결과 : {result['generation']}")